# Reducción de dimensionalidad: selección de variables con RFECV

Este notebook estima cuántas de las **65 variables originales** conviene conservar para una regresión logística. Evalúa todos los tamaños de 65 a 1 mediante dos cortes temporales internos dentro del entrenamiento (préstamos anteriores a 2015). La validación 2015–2016 comprueba las selecciones; la prueba de 2017 en adelante se evalúa solo al final, después de congelar las opciones.

> **Resultados históricos:** esta selección RFECV parte del conjunto anterior de 65 predictoras y sus métricas de prueba ya fueron consultadas. La exploración final actual conserva 59 predictoras con menos del 10% de faltantes. Las cifras de este notebook no validan la reducción sobre el nuevo conjunto ni deben reutilizarse como evaluación final independiente.


## Método elegido y justificación

La meta de este análisis es **elegir cuáles variables originales conservar** para predecir `Charged Off`, manteniendo un modelo interpretable. Por eso usamos RFECV como método principal: la eliminación recursiva ordena variables según el modelo y la validación cruzada compara el rendimiento de distintos tamaños. Aquí el criterio principal es ROC-AUC, calculado con una regresión logística (scikit-learn developers, s. f.). La selección de variables puede ayudar a simplificar el modelo y a entender qué información usa; su utilidad debe comprobarse en los datos y no se garantiza para todo problema (Guyon & Elisseeff, 2003).

PCA no se aplica en este notebook. PCA construye componentes nuevos que combinan variables para conservar la mayor variación posible, mientras que RFECV conserva o elimina predictores originales según su rendimiento para el objetivo (Jolliffe & Cadima, 2016). Como necesitamos comunicar qué campos del préstamo entran al modelo, RFECV responde más directamente a la pregunta del proyecto. Esta razón justifica elegir RFECV para este análisis; no afirma que sea superior a PCA en general.

La versión agrupada es una adaptación para One-Hot Encoding: todos los indicadores derivados de una categórica se tratan como un solo grupo. Así, “38 variables originales” no significa 38 columnas finales; el notebook informa ambas cantidades.


In [1]:
from pathlib import Path
import sys, gc, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.config import (localizar_datos, ANIO_CORTE_VALIDACION, ANIO_CORTE_PRUEBA,
                        UMBRAL_FALTANTES, SEMILLA)
from src.data import inferir_tipos
from src.features import seleccionar_predictoras, preparar_matriz, construir_preprocesador

archivo = localizar_datos()
print("Archivo:", archivo)

Archivo: C:\Users\josue\Downloads\accepted_2007_to_2018Q4.csv.gz


## 1. Cargar datos y mantener los periodos separados

Se leen solo las columnas candidatas al modelo, la etiqueta y la fecha de otorgamiento. Primero identificamos los préstamos con resultado conocido; luego separamos entrenamiento, validación y prueba por año. La prueba no se usa para seleccionar variables ni ajustar el modelo; se evalúa una sola vez en la sección final.

In [2]:
columnas_fuente = pd.read_csv(archivo, nrows=0).columns.tolist()
tipos_inferidos, _ = inferir_tipos(archivo)
candidatas = seleccionar_predictoras(pd.DataFrame(columns=columnas_fuente), [], incluir_precio_lc=False)
columnas_lectura = [c for c in columnas_fuente if c in candidatas + ["loan_status", "issue_d"]]
dtype_lectura = {c: tipos_inferidos[c] for c in columnas_lectura}

inicio = time.perf_counter()
df = pd.read_csv(archivo, usecols=columnas_lectura, dtype=dtype_lectura,
                 skipinitialspace=True, low_memory=False)
for c in ["issue_d", "earliest_cr_line", "sec_app_earliest_cr_line"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c].astype("string"), format="%b-%Y", errors="coerce")
if "term" in df.columns:
    df["term"] = pd.to_numeric(
        df["term"].astype("string").str.strip().str.replace(" months", "", regex=False),
        errors="coerce").astype("Int8")

columnas_vacias = df.columns[df.isna().all()].tolist()
if columnas_vacias:
    df = df.drop(columns=columnas_vacias)
    candidatas = [c for c in candidatas if c not in columnas_vacias]
df = df.loc[df["loan_status"].isin(["Fully Paid", "Charged Off"])].copy()
df["default"] = df["loan_status"].map({"Fully Paid": 0, "Charged Off": 1}).astype("int8")
df["fecha_prestamo"] = df["issue_d"]
df["anio"] = df["issue_d"].dt.year
assert df["anio"].notna().all()

train_df = df.loc[df.anio < ANIO_CORTE_VALIDACION].copy()
valid_df = df.loc[df.anio.between(ANIO_CORTE_VALIDACION, ANIO_CORTE_PRUEBA - 1)].copy()
prueba_filas = int((df.anio >= ANIO_CORTE_PRUEBA).sum())
print(f"Lectura y preparación: {time.perf_counter()-inicio:.1f} s")
print(f"Elegibles: {len(df):,}; entrenamiento: {len(train_df):,}; validación: {len(valid_df):,}; prueba reservada: {prueba_filas:,}")
print("Años en entrenamiento:", sorted(train_df.anio.unique().tolist()))

Lectura y preparación: 48.5 s
Elegibles: 1,345,310; entrenamiento: 451,059; validación: 668,640; prueba reservada: 225,611
Años en entrenamiento: [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014]


## 2. Reconstruir el escenario de 65 variables

Aplicamos la misma regla del análisis anterior: variables con más de 95% de faltantes en el entrenamiento se excluyen. Las fechas de historial se convierten en antigüedad en meses. El preprocesamiento de cada corte se ajusta solo con las filas de entrenamiento de ese corte.

In [3]:
faltantes_train_pct = train_df[candidatas].isna().mean() * 100
columnas_mas95 = faltantes_train_pct[faltantes_train_pct > UMBRAL_FALTANTES].index.tolist()

train_raw_todas = preparar_matriz(train_df[candidatas + ["fecha_prestamo"]]).drop(columns="fecha_prestamo")
valid_raw_todas = preparar_matriz(valid_df[candidatas + ["fecha_prestamo"]]).drop(columns="fecha_prestamo")
mapa_fecha = {
    "earliest_cr_line": "meses_desde_earliest_cr_line",
    "sec_app_earliest_cr_line": "meses_desde_sec_app_earliest_cr_line",
}
excluidas_preparadas = [mapa_fecha.get(c, c) for c in columnas_mas95]
variables65 = [c for c in train_raw_todas.columns if c not in excluidas_preparadas]
train_raw = train_raw_todas[variables65].copy()
valid_raw = valid_raw_todas[variables65].copy()
y_train = train_df["default"].to_numpy()
y_valid = valid_df["default"].to_numpy()

print("Predictoras originales tras la regla de faltantes:", len(variables65))
print("Variables con más de 95% de faltantes excluidas:", len(columnas_mas95))
assert len(variables65) == 65, f"Se esperaban 65 variables; se encontraron {len(variables65)}."

Predictoras originales tras la regla de faltantes: 65
Variables con más de 95% de faltantes excluidas: 30


## 3. Definir validaciones temporales internas

Tomamos los últimos tres años disponibles dentro del entrenamiento como posibles bloques de validación interna. Para cada año, se entrena solo con años anteriores. Si una variable candidata no tiene **ningún dato observado** en el bloque de entrenamiento, ese corte se omite: no se puede medir la utilidad de una variable que todavía no existía en los datos de ese periodo. Esta regla evita contar como seleccionables variables que el preprocesador tendría que descartar por completo.

In [4]:
anios_entrenamiento = sorted(train_df["anio"].unique().tolist())
anios_candidatos_cv = anios_entrenamiento[-3:]
pliegues = []
cortes_omitidos = []
for anio_validacion in anios_candidatos_cv:
    indices_fit = np.flatnonzero(train_df["anio"].to_numpy() < anio_validacion)
    indices_cv = np.flatnonzero(train_df["anio"].to_numpy() == anio_validacion)
    if len(indices_fit) == 0 or len(indices_cv) == 0:
        continue
    sin_datos = train_raw.iloc[indices_fit].columns[train_raw.iloc[indices_fit].isna().all()].tolist()
    if sin_datos:
        cortes_omitidos.append({"año": anio_validacion, "variables_sin_datos": len(sin_datos)})
        continue
    pliegues.append((anio_validacion, indices_fit, indices_cv))

if cortes_omitidos:
    print("Cortes omitidos porque aún no tenían datos de todas las variables:")
    display(pd.DataFrame(cortes_omitidos))
resumen_pliegues = pd.DataFrame([
    {"año_validación": anio, "filas_entrenamiento": len(i_fit),
     "filas_validación": len(i_cv),
     "incumplimiento_entrenamiento": y_train[i_fit].mean(),
     "incumplimiento_validación": y_train[i_cv].mean()}
    for anio, i_fit, i_cv in pliegues
])
display(resumen_pliegues.round(4))
assert len(pliegues) >= 2, "Se necesitan al menos dos cortes temporales internos con datos suficientes."

Cortes omitidos porque aún no tenían datos de todas las variables:
 año  variables_sin_datos
2012                   37
 año_validación  filas_entrenamiento  filas_validación  incumplimiento_entrenamiento  incumplimiento_validación
           2013                93153            134804                        0.1537                     0.1560
           2014               227957            223102                        0.1550                     0.1845


## 4. RFECV agrupado: probar cada cantidad de variables

En cada corte temporal, el modelo se ajusta con años anteriores. Se calcula la fuerza de cada variable original agrupando sus coeficientes One-Hot; después se quita la variable más débil y se vuelve a medir ROC-AUC en el año reservado para ese corte. Repetimos el proceso de **65 a 1 variable**, así se evalúa cada cantidad y no solo saltos de cinco. La validación interna mide cada tamaño, pero no participa en el cálculo de coeficientes ni en el orden de eliminación.

In [5]:
def mapa_columnas_codificadas(preprocesador, variables):
    nombres = preprocesador.get_feature_names_out().tolist()
    fuente = {}
    for nombre in nombres:
        posibles = [v for v in variables if nombre == v or nombre.startswith(v + "_")]
        if not posibles:
            raise ValueError(f"No se pudo asociar la columna codificada {nombre} con su variable original.")
        fuente[nombre] = max(posibles, key=len)
    indices = {
        v: np.array([i for i, nombre in enumerate(nombres) if fuente[nombre] == v], dtype=int)
        for v in variables
    }
    return nombres, indices

def ejecutar_ruta_rfe(X_fit, y_fit, X_eval, y_eval, variables):
    preprocesador, _ = construir_preprocesador(X_fit, con_indicador=[])
    matriz_fit = preprocesador.fit_transform(X_fit)
    matriz_eval = preprocesador.transform(X_eval)
    _, indices_por_variable = mapa_columnas_codificadas(preprocesador, variables)

    activas = list(variables)
    registros = []
    subconjuntos = {}
    while True:
        indices_activos = np.sort(np.concatenate([indices_por_variable[v] for v in activas]))
        modelo = LogisticRegression(C=1.0, max_iter=2000, solver="lbfgs", random_state=SEMILLA)
        modelo.fit(matriz_fit[:, indices_activos], y_fit)
        scores = modelo.predict_proba(matriz_eval[:, indices_activos])[:, 1]
        subconjuntos[len(activas)] = activas.copy()
        registros.append({
            "n_variables": len(activas),
            "columnas_codificadas": len(indices_activos),
            "ROC-AUC": roc_auc_score(y_eval, scores),
            "PR-AUC": average_precision_score(y_eval, scores),
        })
        if len(activas) == 1:
            break

        coef = modelo.coef_[0]
        fuerza = {
            v: float(np.linalg.norm(coef[np.isin(indices_activos, indices_por_variable[v])]))
            for v in activas
        }
        quitar = min(activas, key=lambda v: fuerza[v])
        activas.remove(quitar)

    del matriz_fit, matriz_eval, preprocesador
    gc.collect()
    return pd.DataFrame(registros), subconjuntos

resultados_cv = []
subconjuntos_cv = {}
inicio_rfecv = time.perf_counter()
for anio, indices_fit, indices_cv in pliegues:
    print(f"Procesando corte: entrenamiento hasta {anio-1}, validación {anio}...", flush=True)
    tabla_corte, seleccion_corte = ejecutar_ruta_rfe(
        train_raw.iloc[indices_fit], y_train[indices_fit],
        train_raw.iloc[indices_cv], y_train[indices_cv], variables65
    )
    tabla_corte["año_validación_interna"] = anio
    resultados_cv.append(tabla_corte)
    subconjuntos_cv[anio] = seleccion_corte

resultados_cv = pd.concat(resultados_cv, ignore_index=True)
print(f"RFECV terminado en {time.perf_counter()-inicio_rfecv:.1f} s")

Procesando corte: entrenamiento hasta 2012, validación 2013...
Procesando corte: entrenamiento hasta 2013, validación 2014...
RFECV terminado en 367.9 s


## 5. Comparar cantidades y obtener candidatos

La primera opción es la cantidad con mayor ROC-AUC promedio en los cortes internos. También calculamos la regla heurística de **un error estándar**, que propone un conjunto más pequeño cuyo promedio queda dentro de un error estándar del máximo. La validación 2015–2016 comprueba si ese candidato conserva el rendimiento. PR-AUC se incluye como métrica secundaria.

In [6]:
resumen_cv = (resultados_cv.groupby("n_variables")
              .agg(ROC_AUC_media=("ROC-AUC", "mean"),
                   ROC_AUC_desv=("ROC-AUC", "std"),
                   PR_AUC_media=("PR-AUC", "mean"),
                   PR_AUC_desv=("PR-AUC", "std"),
                   columnas_codificadas_promedio=("columnas_codificadas", "mean"),
                   cortes=("año_validación_interna", "nunique"))
              .sort_index(ascending=False))
resumen_cv["ROC_AUC_error_estandar"] = resumen_cv["ROC_AUC_desv"] / np.sqrt(resumen_cv["cortes"])
numero_mejor = int(resumen_cv["ROC_AUC_media"].idxmax())
puntaje_mejor = float(resumen_cv.loc[numero_mejor, "ROC_AUC_media"])
error_estandar_mejor = float(resumen_cv.loc[numero_mejor, "ROC_AUC_error_estandar"])
umbral_un_error = puntaje_mejor - error_estandar_mejor
numero_un_error = int(resumen_cv.loc[resumen_cv["ROC_AUC_media"] >= umbral_un_error].index.min())

print("Cantidad con mayor ROC-AUC promedio:", numero_mejor)
print("Candidata más pequeña dentro de un error estándar:", numero_un_error)
print("ROC-AUC promedio máximo:", round(puntaje_mejor, 4))
print("Umbral de un error estándar:", round(umbral_un_error, 4))
display(resumen_cv.round(4))

Cantidad con mayor ROC-AUC promedio: 38
Candidata más pequeña dentro de un error estándar: 13
ROC-AUC promedio máximo: 0.694
Umbral de un error estándar: 0.6824
 ROC_AUC_media  ROC_AUC_desv  PR_AUC_media  PR_AUC_desv  columnas_codificadas_promedio  cortes  ROC_AUC_error_estandar
        0.6935        0.0168        0.3117       0.0520                          138.0       2                  0.0119
        0.6935        0.0168        0.3117       0.0520                          138.0       2                  0.0119
        0.6935        0.0168        0.3117       0.0520                          137.0       2                  0.0119
        0.6936        0.0168        0.3117       0.0519                          136.0       2                  0.0119
        0.6936        0.0167        0.3117       0.0520                          135.0       2                  0.0118
        0.6935        0.0168        0.3116       0.0520                          134.0       2                  0.0119
      

## 6. Estabilidad de las variables seleccionadas

Comprobamos cuántas veces aparece cada variable en la selección de cada corte interno al tamaño recomendado por un error estándar. Una variable seleccionada en todos los cortes es más estable que otra que aparece solo en uno. Esto complementa la métrica: un buen promedio con selecciones muy distintas merece cautela.

In [7]:
seleccion_por_corte = {
    anio: subconjuntos_cv[anio][numero_un_error]
    for anio, _, _ in pliegues
}
frecuencia = pd.Series(
    [v for conjunto in seleccion_por_corte.values() for v in conjunto]
).value_counts()
tabla_estabilidad = pd.DataFrame({
    "variable": variables65,
    "veces_seleccionada": [int(frecuencia.get(v, 0)) for v in variables65],
})
tabla_estabilidad["proporcion_cortes"] = tabla_estabilidad["veces_seleccionada"] / len(pliegues)
tabla_estabilidad = tabla_estabilidad.sort_values(
    ["veces_seleccionada", "variable"], ascending=[False, True]).reset_index(drop=True)
print("Variables seleccionadas por corte temporal:")
for anio, conjunto in seleccion_por_corte.items():
    print(anio, len(conjunto), "variables")
display(tabla_estabilidad)

Variables seleccionadas por corte temporal:
2013 13 variables
2014 13 variables
                      variable  veces_seleccionada  proporcion_cortes
                    addr_state                   2                1.0
                    annual_inc                   2                1.0
                    emp_length                   2                1.0
               fico_range_high                   2                1.0
                fico_range_low                   2                1.0
                home_ownership                   2                1.0
                inq_last_6mths                   2                1.0
                     loan_amnt                   2                1.0
               num_actv_rev_tl                   2                1.0
           num_rev_tl_bal_gt_0                   2                1.0
                       purpose                   2                1.0
                          term                   2                1.0
          

## 7. Reajustar en todo el entrenamiento y comprobar en validación

Una vez elegida la cantidad por los cortes internos, repetimos la eliminación variable por variable usando todo el entrenamiento anterior a 2015 para obtener los subconjuntos finales. Luego comparamos 65, 30, la cantidad de mayor ROC-AUC interno y la candidata de un error estándar en 2015–2016. Reportamos tanto variables originales como columnas One-Hot. La prueba se mantiene fuera de esta selección.

In [8]:
# Ruta de eliminación sobre todo el periodo de entrenamiento, de 65 a 1.
pre_final, _ = construir_preprocesador(train_raw, con_indicador=[])
matriz_train_final = pre_final.fit_transform(train_raw)
matriz_valid_final = pre_final.transform(valid_raw)
_, indices_final = mapa_columnas_codificadas(pre_final, variables65)

activas_final = list(variables65)
selecciones_finales = {len(activas_final): activas_final.copy()}
while len(activas_final) > 1:
    idx = np.sort(np.concatenate([indices_final[v] for v in activas_final]))
    modelo_ranking = LogisticRegression(C=1.0, max_iter=2000, solver="lbfgs", random_state=SEMILLA)
    modelo_ranking.fit(matriz_train_final[:, idx], y_train)
    coef = modelo_ranking.coef_[0]
    fuerza = {
        v: float(np.linalg.norm(coef[np.isin(idx, indices_final[v])]))
        for v in activas_final
    }
    quitar = min(activas_final, key=lambda v: fuerza[v])
    activas_final.remove(quitar)
    selecciones_finales[len(activas_final)] = activas_final.copy()

# Compara cantidades originales y cuenta las columnas reales tras One-Hot.
conteos_a_comparar = sorted(set([65, 30, numero_mejor, numero_un_error]), reverse=True)
filas_validacion = []
for cantidad in conteos_a_comparar:
    variables = selecciones_finales[cantidad]
    idx = np.sort(np.concatenate([indices_final[v] for v in variables]))
    modelo = LogisticRegression(C=1.0, max_iter=2000, solver="lbfgs", random_state=SEMILLA)
    modelo.fit(matriz_train_final[:, idx], y_train)
    score = modelo.predict_proba(matriz_valid_final[:, idx])[:, 1]
    filas_validacion.append({
        "variables_originales": cantidad,
        "columnas_codificadas": len(idx),
        "ROC-AUC_validación": roc_auc_score(y_valid, score),
        "PR-AUC_validación": average_precision_score(y_valid, score),
        "variables_seleccionadas": variables,
    })
tabla_validacion = pd.DataFrame(filas_validacion)
display(tabla_validacion.drop(columns="variables_seleccionadas").round(4))
print("Prueba desde 2017 en adelante: reservada, no evaluada.")

 variables_originales  columnas_codificadas  ROC-AUC_validación  PR-AUC_validación
                   65                   139              0.7156             0.4093
                   38                   113              0.7151             0.4087
                   30                   105              0.7139             0.4081
                   13                    87              0.7085             0.4001
Prueba desde 2017 en adelante: reservada, no evaluada.


## 8. Mostrar las variables candidatas y revisar correlaciones

Mostramos las listas seleccionadas para la cantidad óptima por ROC-AUC interno, 30 variables y la candidata de un error estándar. Después listamos pares numéricos muy correlacionados para revisar posibles redundancias; no se eliminan automáticamente.

In [9]:
cantidades_lista = sorted(set([numero_mejor, 30, numero_un_error]), reverse=True)
for cantidad in cantidades_lista:
    print(f"{cantidad} variables originales; {len(selecciones_finales[cantidad])} seleccionadas:")
    display(pd.DataFrame({"variable_original": selecciones_finales[cantidad]}))

# Diagnóstico exploratorio de correlación numérica en una muestra del entrenamiento.
numericas = train_raw.select_dtypes(include="number").columns.tolist()
tamano_corr = min(100_000, len(train_raw))
indices_corr = np.random.default_rng(SEMILLA).choice(len(train_raw), size=tamano_corr, replace=False)
corr = train_raw.iloc[indices_corr][numericas].corr(method="spearman").abs()
pares = []
for i, a in enumerate(numericas):
    for b in numericas[i+1:]:
        valor = corr.loc[a, b]
        if pd.notna(valor) and valor >= 0.90:
            pares.append({"variable_1": a, "variable_2": b, "correlación_absoluta": valor})
tabla_pares = pd.DataFrame(pares).sort_values("correlación_absoluta", ascending=False) if pares else pd.DataFrame(columns=["variable_1", "variable_2", "correlación_absoluta"])
print(f"Pares numéricos con |Spearman| >= 0.90 en muestra de {tamano_corr:,} filas:")
display(tabla_pares.head(30).round(3))

38 variables originales; 38 seleccionadas:
           variable_original
                   loan_amnt
                        term
                  emp_length
              home_ownership
                  annual_inc
         verification_status
                     purpose
                  addr_state
                         dti
                 delinq_2yrs
              fico_range_low
             fico_range_high
              inq_last_6mths
                  revol_util
                   total_acc
                 tot_cur_bal
            total_rev_hi_lim
        acc_open_past_24mths
                 avg_cur_bal
              bc_open_to_buy
                     bc_util
        mo_sin_old_rev_tl_op
                    mort_acc
        mths_since_recent_bc
              num_actv_bc_tl
             num_actv_rev_tl
                 num_bc_sats
                   num_bc_tl
               num_rev_accts
         num_rev_tl_bal_gt_0
          num_tl_op_past_12m
              pct_tl_nvr_dlq


## 9. Evaluación final en prueba (2017 en adelante)

Las opciones 65, 38 y 30 se definieron antes de medir este conjunto. Para cada modelo elegimos en la validación 2015–2016 el umbral que maximiza F1; después calculamos ROC-AUC, Average Precision, precisión, recall, F1 y matriz de confusión en 2017 en adelante. La prueba no participa en la selección ni en el preprocesamiento. Esta evaluación queda consumida para esta comparación.

In [10]:
from src.evaluate import elegir_umbral, evaluar

test_df = df.loc[df["anio"] >= ANIO_CORTE_PRUEBA].copy()
test_raw_todas = preparar_matriz(
    test_df[candidatas + ["fecha_prestamo"]]
).drop(columns="fecha_prestamo")
test_raw = test_raw_todas[variables65]
y_test = test_df["default"].to_numpy()

# El preprocesador se ajustó solo con entrenamiento.
matriz_test_final = pre_final.transform(test_raw)
filas_prueba = []
for cantidad in [65, 38, 30]:
    variables = variables65 if cantidad == 65 else selecciones_finales[cantidad]
    idx = np.sort(np.concatenate([indices_final[v] for v in variables]))
    modelo = LogisticRegression(C=1.0, max_iter=2000, solver="lbfgs", random_state=SEMILLA)
    modelo.fit(matriz_train_final[:, idx], y_train)

    # El umbral se elige en validación maximizando F1, nunca con la prueba.
    score_validacion = modelo.predict_proba(matriz_valid_final[:, idx])[:, 1]
    umbral = elegir_umbral(y_valid, score_validacion, criterio="f1")
    score_prueba = modelo.predict_proba(matriz_test_final[:, idx])[:, 1]
    metricas = evaluar(y_test, score_prueba, umbral)
    filas_prueba.append({
        "variables_originales": cantidad,
        "columnas_codificadas": len(idx),
        "umbral_elegido_validacion": metricas["umbral"],
        "ROC-AUC_prueba": metricas["roc_auc"],
        "Average_Precision_prueba": metricas["pr_auc"],
        "Precision_prueba": metricas["precision"],
        "Recall_prueba": metricas["recall"],
        "F1_prueba": metricas["f1"],
        "VN": metricas["matriz_confusion"]["vn"],
        "FP": metricas["matriz_confusion"]["fp"],
        "FN": metricas["matriz_confusion"]["fn"],
        "VP": metricas["matriz_confusion"]["vp"],
    })

tabla_prueba = pd.DataFrame(filas_prueba)
print(f"Préstamos en prueba: {len(test_df):,}")
print("Distribución del objetivo:")
display(test_df["loan_status"].value_counts().rename_axis("loan_status").to_frame("cantidad"))
display(tabla_prueba.round(4))

Préstamos en prueba: 225,611
Distribución del objetivo:
    loan_status  cantidad
     Fully Paid    177596
    Charged Off     48015
 variables_originales  columnas_codificadas  umbral_elegido_validacion  ROC-AUC_prueba  Average_Precision_prueba  Precision_prueba  Recall_prueba  F1_prueba     VN     FP    FN    VP
                   65                   139                    0.1800          0.6910                    0.3577            0.3391         0.5666     0.4243 124574  53022 20809 27206
                   38                   113                    0.1788          0.6902                    0.3567            0.3374         0.5662     0.4229 124211  53385 20827 27188
                   30                   105                    0.1797          0.6891                    0.3559            0.3363         0.5668     0.4221 123875  53721 20800 27215


## Referencias

Guyon, I., & Elisseeff, A. (2003). An introduction to variable and feature selection. *Journal of Machine Learning Research, 3*, 1157–1182. https://www.jmlr.org/papers/volume3/guyon03a/guyon03a.pdf

Jolliffe, I. T., & Cadima, J. (2016). Principal component analysis: A review and recent developments. *Philosophical Transactions of the Royal Society A: Mathematical, Physical and Engineering Sciences, 374*(2065), 20150202. https://doi.org/10.1098/rsta.2015.0202

scikit-learn developers. (s. f.). *RFECV: Recursive feature elimination with cross-validation*. scikit-learn documentation. https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFECV.html


## Conclusión y recomendación

Al evaluar cada tamaño de 65 a 1 variable original, RFECV obtuvo su mayor ROC-AUC promedio interno con **38 variables originales**, que generan **113 columnas codificadas**. Es el tamaño con mejor promedio en los dos cortes temporales internos, no una garantía de óptimo universal.

En la evaluación final de prueba (2017 en adelante), el baseline con 65 variables obtuvo ROC-AUC **0.6910** y PR-AUC **0.3577**; RFECV con 38 obtuvo **0.6902** y **0.3567**; la selección de 30 obtuvo **0.6891** y **0.3559**. El baseline tuvo el resultado más alto, y las diferencias fueron pequeñas. RFECV 38 redujo las columnas codificadas de 139 a 113 y mantuvo un rendimiento cercano.

La prueba contiene 225,611 préstamos: 177,596 Fully Paid y 48,015 Charged Off. Como ya se usó para esta comparación final, no se debe reajustar el modelo mirando estos resultados y luego presentar la misma prueba como evaluación independiente.

Con los umbrales elegidos en validación para maximizar F1, el modelo de 65 obtuvo precisión **0.3391** y recall **0.5666** en prueba. El de 38 obtuvo **0.3374** y **0.5662**; el de 30, **0.3363** y **0.5668**. La matriz de confusión confirma que, con este criterio, se detecta alrededor del 57% de los incumplimientos y cerca de un tercio de las alertas son verdaderos incumplimientos. Un umbral más alto suele subir precisión y bajar recall; su elección dependería del costo relativo de falsas alarmas y casos no detectados.